# Full Empirical Grid Search

This notebook is used to attempt a parameter recovery on the full published empirical data.

In [1]:
from ast import literal_eval
import pandas as pd

# 1. Load data
df_raw = pd.read_csv('1ms_trial_data.csv')

# 2. Drop nuisance trials
to_drop = pd.read_csv("dropped_trials.csv").rename(columns={"parcode": "sub_id"})

df = df_raw.loc[
    ~df_raw.set_index(["sub_id", "trial"]).index.isin(
        to_drop.set_index(["sub_id", "trial"]).index
    )
    & (~df_raw["hidden"])
]

# 3. Adjustments
df['RT'] = (df['RT']*1000).astype(int) # RT unit scaling
df['fixation'] = df['fixation'].apply(literal_eval) # String to list as a result of csv saving
df['choice'] = df['choice'].replace({"left": 0, "right": 1}) # Map choice to 0 or 1

/var/folders/59/03h51wmn6xn1jhvb7kj9fjc40000gn/T/ipykernel_29913/4055317967.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['RT'] = (df['RT']*1000).astype(int) # RT unit scaling
/var/folders/59/03h51wmn6xn1jhvb7kj9fjc40000gn/T/ipykernel_29913/4055317967.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['fixation'] = df['fixation'].apply(literal_eval) # String to list as a result of csv saving
/var/folders/59/03h51wmn6xn1jhvb7kj9fjc40000gn/T/ipykernel_29913/4055317967.py:20: FutureWarning: Down

In [2]:
import pyddm

dt = 0.01
model_conditions = {'drift_rate': 0.3, 'theta': 0.5, 'noise': 0.6}

empirical_sample_df = df.loc[:, ['avgWTP_left', 'avgWTP_right','fixation','RT','choice']]
empirical_sample_df['RT'] = empirical_sample_df['RT']/1000

sample = pyddm.Sample.from_pandas_dataframe(
    empirical_sample_df,
    choice_column_name="choice",
    rt_column_name="RT",
    choice_names=("left", "right")
)

print(f'Average empirical RT: {empirical_sample_df["RT"].mean():.2f} seconds (out of {len(empirical_sample_df)} trials)')
empirical_sample_df.head()

Average empirical RT: 1.39 seconds (out of 9302 trials)


,avgWTP_left,avgWTP_right,fixation,RT,choice
100,5.00,1.00,"(4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...",0.940,0
101,1.25,1.00,"(4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...",1.147,0
102,5.00,1.00,"(4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...",0.818,0
103,1.25,4.25,"(4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...",0.796,1
104,4.00,3.75,"(4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...",1.313,1


In [7]:
import itertools
import numpy as np

d_vals = np.linspace(0.05, 0.45, 9)
noise_vals = np.linspace(0.4, 0.7, 7)

grid = list(itertools.product(d_vals, noise_vals))

In [11]:
from pyddm.models.loss import LossLikelihood

results = []

def drift_function(x, t, d, avgWTP_left, avgWTP_right, fixation):
    idx = min(int(t / 0.001), len(fixation) - 1)
    f = fixation[idx]
    theta = model_conditions['theta'] # For theta as a constant
    if f == 0:   # no fixation
        drift_val =  0
    elif f == 1:
        drift_val = d * (avgWTP_left - theta * avgWTP_right)
    else:
        drift_val = d * (theta * avgWTP_left - avgWTP_right)
    
    return np.ones_like(x) * drift_val

def noise_function(x, t, noise):
    return np.ones_like(x) * noise

for d_val, noise_val in grid:
    m = pyddm.gddm(
        drift=drift_function,
        noise=noise_function,
        parameters={"d": d_val, "noise": noise_val},
        conditions=["avgWTP_left","avgWTP_right","fixation"],
        choice_names=("left","right"),
        T_dur=23,
        dx=dt,
        dt=dt
    )

    lossfunc = LossLikelihood(sample=sample, model=m, dt=m.dt, T_dur=m.T_dur)
    print(f"Running params [{d_val}, {noise_val}]...")
    nll = lossfunc.loss(m)
    print(f"NLL = {nll}")
    results.append({"drift": d_val, "noise": noise_val, "nll": nll})

Running params [0.05, 0.4]...
NLL = 32073.159090832254
Running params [0.05, 0.45]...
NLL = 27217.45196155206
Running params [0.05, 0.5]...


KeyboardInterrupt: 